# QDebug demo

Attach full lifecycle visibility to ordinary Qiskit code with one activation call.
Everything below the activation cell is plain Qiskit — no QDebug-specific calls
until we read the timeline back at the end.

In [ ]:
import qdebug

report = qdebug.activate()
report.layers

## Ordinary Qiskit code

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(3, 3, name="ghz")
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure(range(3), range(3))

backend = AerSimulator()
transpiled = transpile(qc, basis_gates=["cx", "u"], optimization_level=1)
counts = backend.run(transpiled, shots=1000).result().get_counts()
counts

## What QDebug saw

Construction, transpilation and execution all landed on one timeline for `qc`,
even though the circuit that actually ran was the transpiled copy.

In [ ]:
timeline = qdebug.timeline(qc)
print(timeline.summary())

In [ ]:
for event in timeline.events:
    print(f"[{event.seq:>3}] {event.type.value:<22} {event.describe()}")

## State evolution

The snapshots are real simulator statevectors captured during the run you
already made, then stripped from the result object you received.

In [ ]:
for snapshot in timeline.snapshots:
    probabilities = {
        format(i, f"0{snapshot.num_qubits}b"): round(p, 3)
        for i, p in enumerate(snapshot.probabilities())
        if p > 1e-9
    }
    print(f"step {snapshot.step} ({snapshot.gate or 'initial':<7}) {probabilities}")

## Interactive timeline widget

Requires the viz extra: `pip install "qdebug[viz]"` (ipywidgets). A slider
walks every event, drawing the circuit as it existed at that moment and the
captured statevectors.

In [ ]:
timeline.show()

In [ ]:
qdebug.deactivate()